In [1]:
import plotly.graph_objs as go
import pandas as pd

In [ ]:
class Visualizer:
    """Handles all plotting logic including daily profiles, seasonal trends, and box plots."""
    
    @staticmethod
    def get_colours():
        """Returns colour order."""
        return [
            '#000000', '#E69F00', '#2CA02C', "#3827D6", 
            '#9467BD', '#8C564B', '#E377C2', '#7F7F7F', 
            "#DB1212", '#17BECF', "#E600C0", '#56B4E9'
        ]

    @classmethod
    def plot_all(cls, dfs, y_label, data_labels, title):
        """The base plotting engine used by other functions."""
        width_cm = 32
        height_cm = 12
        PIXEL_PER_CM = 37.8
        colours = cls.get_colours()
        fig = go.Figure()
        
        for i in range(len(dfs)):
            fig.add_trace(go.Scatter(
                x=dfs[i].index, 
                y=dfs[i], 
                mode='lines', 
                name=data_labels[i], 
                opacity=0.6, 
                line_color=colours[i % len(colours)],
                line=dict(width=3.5) 
            ))
        
        fig.update_layout(
            width=width_cm * PIXEL_PER_CM,
            height=height_cm * PIXEL_PER_CM,
            autosize=False,
            title=dict(text=title, font=dict(size=26)),
            xaxis=dict(title=dict(text='Time'), tickfont=dict(size=18), gridcolor='lightgrey'),
            yaxis=dict(title=dict(text=y_label, font=dict(size=22)), tickfont=dict(size=18), gridcolor='lightgrey'),
            legend=dict(font=dict(size=16), bgcolor='rgba(255,255,255,0.5)'),
            template='plotly_white',
            margin=dict(l=100, r=50, t=100, b=100)
        )
        return fig

    @classmethod
    def plot_daily_profile(cls, monthly_matrix, group, title):
        """Standard daily profile plot."""
        PIXEL_PER_CM = 37.8
        width_cm = 29.57
        height_cm = 15.1
        
        month_data = [monthly_matrix[m] for m in monthly_matrix.columns]
        fig = cls.plot_all(dfs=month_data, y_label='Average Power [kW]', data_labels=monthly_matrix.columns, title=title)
        fig.update_traces(line=dict(width=3.5)) 

        peak_values = monthly_matrix.max()
        peak_times = monthly_matrix.idxmax()
        colours = cls.get_colours()
        
        raw_annos = []
        for month in group:
            if month in peak_times and month in peak_values:
                m_time = peak_times[month]
                current_mins = m_time.hour * 60 + m_time.minute
                month_idx = list(monthly_matrix.columns).index(month)
                raw_annos.append({
                    'month': month, 'x_val': m_time, 'mins': current_mins,
                    'y': peak_values[month], 'color': colours[month_idx % len(colours)]
                })

        raw_annos.sort(key=lambda x: x['y'])
        annotations = []
        for anno in raw_annos:
            x_offset, text_anchor = 0, 'center'
            for placed in annotations:
                x_dist = abs(anno['mins'] - placed['mins_tracker'])
                y_dist = abs(anno['y'] - placed['y'])
                if x_dist < 90 and y_dist < (max(peak_values) * 0.05):
                    if anno['mins'] >= placed['mins_tracker']:
                        x_offset, text_anchor = 45, 'left'
                    else:
                        x_offset, text_anchor = -45, 'right'

            annotations.append(dict(
                x=anno['x_val'], y=anno['y'], mins_tracker=anno['mins'],
                text=f"<b>{anno['month'][:100]}</b>", showarrow=False,
                font=dict(size=18, color=anno['color']), xanchor=text_anchor,
                xshift=x_offset, yanchor='middle', bgcolor="rgba(255, 255, 255, 0.5)"
            ))

        all_times = monthly_matrix.index.tolist()
        tick_vals = [t for t in all_times if t.hour % 4 == 0 and t.minute == 0]
        fig.update_layout(
            width=width_cm * PIXEL_PER_CM, height=height_cm * PIXEL_PER_CM,
            annotations=[{k: v for k, v in a.items() if k != 'mins_tracker'} for a in annotations],
            xaxis=dict(tickmode='array', tickvals=tick_vals, ticktext=[f"{t.hour:02d}:00" for t in tick_vals], tickfont=dict(size=20))
        )
        return fig
    @classmethod
    def get_month_order(cls):
        """Returns the order of months."""
        return ['January', 'February', 'March', 'April', 'May', 'June', 
                'July', 'August', 'September', 'October', 'November', 'December']
    @classmethod
    def get_monthly_matrix(cls, df, column, time_freq):
        """
        Turns DEOP data into monthly averages in time_freq
        """
        months_order = cls.get_month_order()
        
        df = df.copy()
        
        df['Month'] = df.index.month_name()
        df[time_freq] = df.index.floor(time_freq).time 
        
        grouped = df.groupby(['Month', time_freq])[[column]].mean()
        
        grouped = grouped.reindex(months_order, level='Month')
        
        return grouped[column].unstack(level='Month')
    @classmethod
    def plot_seasonal_profiles(cls, df, column, title):
        """Calculates and plots seasonal averages."""
        monthly_matrix = cls.get_monthly_matrix(df, column, '60min')

        seasons = pd.DataFrame()
        seasons['Summer'] = monthly_matrix[['June', 'July', 'August']].mean(axis=1)
        seasons['Autumn'] = monthly_matrix[['September', 'October', 'November']].mean(axis=1)
        
        if all(m in monthly_matrix.columns for m in ['December', 'January', 'February']):
            seasons['Winter'] = monthly_matrix[['December', 'January', 'February']].mean(axis=1) 
            seasons['Spring'] = monthly_matrix[['March', 'April', 'May']].mean(axis=1)
        else:
            seasons['Spring'] = monthly_matrix[['April', 'May']].mean(axis=1)
            seasons['Winter'] = monthly_matrix[['December']].mean(axis=1)
            
        return cls.plot_daily_profile(seasons, ['Spring', 'Summer', 'Autumn', 'Winter'], title)
    @classmethod
    def calculate_daily(cls, deop, column, expected=None):
        """
        Resamples power data to daily totals. 
        """
        daily = pd.concat([expected, deop[column]], axis=1).resample('D').sum()
        
        if expected is None:
            return daily
            
        daily.columns = ['expected', 'actual']

        daily['performance_ratio'] = daily['actual'] / daily['expected']
        
        daily = daily[daily['expected'] > 0]

        return daily
    @classmethod
    def plot_monthly_box(cls, df, column, title, noZero=False):
        """Generates a box plot of daily totals grouped by month."""
        months_order = cls.get_month_order()
        df = cls.calculate_daily(df, column)
        if noZero:
            df = df[df[column] > 0]
        df['Month'] = df.index.month_name()
        fig = go.Figure()
        fig.add_trace(go.Box(
            x=df['Month'],
            y=df[column],
            name='True Generation'
        ))
        fig.update_layout(
            title=title,
            yaxis_title='Total Daily Generation',
            xaxis=dict(categoryorder='array', categoryarray=months_order)
        )
        return fig
    
    @staticmethod
    def calculate_monthly_r2(y_true, y_pred, start_date):
        """
        Calculates and plots R2 scores for every month.
        """
        from plotly.subplots import make_subplots
        from sklearn.metrics import r2_score, mean_absolute_error
        results = pd.DataFrame({
        'Actual': y_true,
        'Predicted': y_pred
        })
        
        results = results.loc[start_date:]
        monthly_r2 = results.resample('ME').apply(lambda x: r2_score(x['Actual'], x['Predicted']))
        monthly_mae = results.resample('ME').apply(lambda x: mean_absolute_error(x['Actual'], x['Predicted']))
        month_names = [d.month_name() for d in monthly_r2.index]
        r2_values = monthly_r2.values
        mae_values = monthly_mae.values

        fig = make_subplots(specs=[[{"secondary_y": True}]])
        fig.add_trace(go.Bar(
            x=month_names,
            y=r2_values,
            name="R² Score",
            text=[f"{v:.2f}" for v in r2_values],
            textposition='outside',
            marker_color="#E69F00",
            width=0.6
        ), secondary_y=False)

       
        fig.add_trace(go.Scatter(
            x=month_names,
            y=mae_values,
            name="MAE (kW)",
            mode='lines+markers+text',
            text=[f"{v:.1f}" for v in mae_values],
            textposition='top center',
            line=dict(color='#000000', width=3),
            marker=dict(size=8)
        ), secondary_y=True)

        fig.update_layout(
            title=dict(text='Monthly Accuracy (R²) vs Mean Error (MAE)', font=dict(size=26), x=0.5),
            template='plotly_white',
            width=1000,
            height=650,
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
            margin=dict(l=100, r=100, t=120, b=100)
        )


        fig.update_yaxes(title_text="R² Score", range=[0, 1.2], secondary_y=False, 
                         tickfont=dict(size=18), title_font=dict(size=22), gridcolor='lightgrey')
        fig.update_yaxes(title_text="MAE (kW)", secondary_y=True, showgrid=False, 
                         tickfont=dict(size=18), title_font=dict(size=22))
        fig.update_xaxes(tickfont=dict(size=18), showgrid=False)

        fig.show()

    @staticmethod
    def plot_model_stats(actuals, predictions, title="Model Statistics"):
        """
        Calculates and plots R2 and MAE for different models.
        """
        from sklearn.metrics import r2_score, mean_absolute_error
        from plotly.subplots import make_subplots
        import plotly.graph_objects as go
        import pandas as pd
        import numpy as np
        
        models = []
        r2_scores = []
        mae_scores = []
        
        for title, preds in predictions.items():
            df = pd.DataFrame({'Actual': actuals, 'Predicted': preds}).dropna()
            r2 = r2_score(df['Actual'], df['Predicted'])
            mae = mean_absolute_error(df['Actual'], df['Predicted'])
                
            models.append(title)
            r2_scores.append(r2)
            mae_scores.append(mae)

        fig = make_subplots(specs=[[{"secondary_y": True}]])
        
        fig.add_trace(go.Bar(
            x=models,
            y=r2_scores,
            name="R² Score",
            text=[f"{v:.3f}" for v in r2_scores],
            textposition='outside',
            marker_color="#E69F00", 
            width=0.4
        ), secondary_y=False)


        fig.add_trace(go.Scatter(
            x=models,
            y=mae_scores,
            name="MAE (kW)",
            mode='lines+markers+text',
            text=[f"{v:.1f}" for v in mae_scores],
            textposition='top center',
            line=dict(color='#000000', width=3),
            marker=dict(size=10)
        ), secondary_y=True)

        fig.update_layout(
            title=dict(text=title, font=dict(size=26), x=0.5),
            template='plotly_white',
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        )

        fig.update_yaxes(title_text="R² Score", range=[0, 1.2], secondary_y=False, gridcolor='lightgrey')
        
        fig.update_yaxes(title_text="MAE (kW)", secondary_y=True)
        
        return fig